In [ ]:
import sqlite3
import os

DB_FILENAME = "financial_data.db"

# Remove existing DB if running fresh
if os.path.exists(DB_FILENAME):
    os.remove(DB_FILENAME)

conn = sqlite3.connect(DB_FILENAME)
cursor = conn.cursor()

# 1. Companies Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS companies (
    company_id INTEGER PRIMARY KEY,
    ticker TEXT UNIQUE NOT NULL,
    name TEXT NOT NULL,
    sector TEXT NOT NULL
);
""")

# 2. Quarterly Financials Table (Foreign Key to companies)
cursor.execute("""
CREATE TABLE IF NOT EXISTS quarterly_financials (
    financial_id INTEGER PRIMARY KEY,
    company_id INTEGER NOT NULL,
    quarter TEXT NOT NULL,
    fiscal_year INTEGER NOT NULL,
    revenue_usd REAL NOT NULL,
    net_income_usd REAL NOT NULL,
    operating_expense_usd REAL NOT NULL,
    FOREIGN KEY (company_id) REFERENCES companies (company_id)
);
""")

# 3. Credit Risk Assessments Table (Foreign Key to companies)
cursor.execute("""
CREATE TABLE IF NOT EXISTS credit_risk (
    risk_id INTEGER PRIMARY KEY,
    company_id INTEGER NOT NULL,
    rating_agency TEXT NOT NULL,
    credit_score INTEGER NOT NULL,
    risk_category TEXT NOT NULL,
    default_probability REAL NOT NULL,
    FOREIGN KEY (company_id) REFERENCES companies (company_id)
);
""")

# Seed Companies
companies = [
    (1, "NVST", "Novastone Logistics", "Industrials"),
    (2, "APEX", "Apex Cloud Systems", "Technology"),
    (3, "VNTG", "Vantage Retail Group", "Consumer Goods")
]
cursor.executemany("INSERT INTO companies VALUES (?, ?, ?, ?)", companies)

# Seed Financials (Q1 - Q3 2025)
financials = [
    # Novastone Logistics
    (1, 1, "Q1", 2025, 4500000.0, 620000.0, 3200000.0),
    (2, 1, "Q2", 2025, 4800000.0, 710000.0, 3350000.0),
    (3, 1, "Q3", 2025, 5100000.0, 830000.0, 3400000.0),
    # Apex Cloud Systems
    (4, 2, "Q1", 2025, 12000000.0, 2400000.0, 8100000.0),
    (5, 2, "Q2", 2025, 13500000.0, 2900000.0, 8900000.0),
    (6, 2, "Q3", 2025, 14200000.0, 3100000.0, 9200000.0),
    # Vantage Retail Group
    (7, 3, "Q1", 2025, 8200000.0, 310000.0, 7200000.0),
    (8, 3, "Q2", 2025, 7900000.0, -120000.0, 7400000.0),
    (9, 3, "Q3", 2025, 8600000.0, 410000.0, 7500000.0)
]
cursor.executemany("INSERT INTO quarterly_financials VALUES (?, ?, ?, ?, ?, ?, ?)", financials)

# Seed Credit Risk
risks = [
    (1, 1, "Moody's", 720, "Low Risk", 0.02),
    (2, 2, "S&P", 810, "Minimal Risk", 0.005),
    (3, 3, "Fitch", 590, "Moderate Risk", 0.085)
]
cursor.executemany("INSERT INTO credit_risk VALUES (?, ?, ?, ?, ?, ?)", risks)

conn.commit()
conn.close()

print(f"Database '{DB_FILENAME}' created and seeded successfully!")

In [ ]:
import sqlite3
import pandas as pd
from pandas.errors import DatabaseError

def execute_safe_query(query: str, db_path: str = "financial_data.db"):
    """
    Executes a SQL query in strict read-only mode.
    Returns the result as a clean pandas DataFrame or a security notice.
    """
    uri = f"file:{db_path}?mode=ro"
    
    try:
        conn = sqlite3.connect(uri, uri=True)
        # Defense-in-depth: prevents any modification statements
        conn.execute("PRAGMA query_only = ON;")
        
        df = pd.read_sql_query(query, conn)
        conn.close()
        return df
    except (sqlite3.OperationalError, DatabaseError, Exception) as e:
        return f"[SECURITY GUARDRAIL TRIGGERED]: {e}"

In [ ]:
# 3-Table Join: Fetch company name, credit score, risk category, and Q3 2025 revenue
test_sql = """
SELECT 
    c.name AS company_name,
    c.sector,
    r.rating_agency,
    r.credit_score,
    r.risk_category,
    q.quarter,
    q.revenue_usd,
    q.net_income_usd
FROM companies c
JOIN credit_risk r ON c.company_id = r.company_id
JOIN quarterly_financials q ON c.company_id = q.company_id
WHERE q.quarter = 'Q3' AND q.fiscal_year = 2025;
"""

result = execute_safe_query(test_sql)
result

In [ ]:
malicious_sql = "DROP TABLE credit_risk;"
security_test = execute_safe_query(malicious_sql)
print(security_test)

# B. Learning and Development Experiments

This section contains earlier implementations and experiments used to develop the Financial Decision Intelligence Assistant.

These cells are retained for learning and reference. They are separate from the final application implementation.

In [ ]:
# B1 — Prototype: Gemini Client and SQL Generation


import os
import re
from google import genai
from google.genai import types

# 1. Set your API Key
os.environ["GEMINI_API_KEY"] = "YOURAPIKEY"

client = genai.Client()

# 2. Schema definition to ground the LLM
DB_SCHEMA_CONTEXT = """
Database Schema:
1. companies (
    company_id INTEGER PRIMARY KEY,
    ticker TEXT UNIQUE NOT NULL,
    name TEXT NOT NULL,
    sector TEXT NOT NULL
);

2. quarterly_financials (
    financial_id INTEGER PRIMARY KEY,
    company_id INTEGER NOT NULL REFERENCES companies(company_id),
    quarter TEXT NOT NULL, -- e.g. 'Q1', 'Q2', 'Q3'
    fiscal_year INTEGER NOT NULL, -- e.g. 2025
    revenue_usd REAL NOT NULL,
    net_income_usd REAL NOT NULL,
    operating_expense_usd REAL NOT NULL
);

3. credit_risk (
    risk_id INTEGER PRIMARY KEY,
    company_id INTEGER NOT NULL REFERENCES companies(company_id),
    rating_agency TEXT NOT NULL, -- e.g. 'Moody\'s', 'S&P', 'Fitch'
    credit_score INTEGER NOT NULL,
    risk_category TEXT NOT NULL, -- e.g. 'Low Risk', 'Minimal Risk', 'Moderate Risk'
    default_probability REAL NOT NULL
);
"""

SYSTEM_PROMPT = f"""
You are a specialized SQL assistant for a financial analytics database.
Given a user's question, output ONLY a valid SQLite read-only query using the provided schema.

Rules:
1. Return ONLY the raw SQL query. Do not wrap it in markdown formatting (no ```sql or ```).
2. Do not include any explanations, preambles, or postambles.
3. Only write SELECT statements. Never write INSERT, UPDATE, DELETE, DROP, or ALTER.
4. Correctly perform JOIN operations on `company_id` when comparing cross-table attributes.

{DB_SCHEMA_CONTEXT}
"""

import time
from google.genai.errors import ServerError, ClientError

# List of models to try in order if one experiences high demand / 503
CANDIDATE_MODELS = [
    "gemini-3.5-flash-lite",
    "gemini-2.5-pro",
    "gemini-embedding_1",
    "gemini-3.8-flash"
]

def generate_sql(question: str) -> str:
    """
    Translates a natural language question into an executable SQL query.
    Includes automated fallback across models if high demand (503) occurs.
    """
    last_error = None

    for model_name in CANDIDATE_MODELS:
        for attempt in range(2):  # Try twice per model with a short pause
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=question,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        temperature=0.0,
                    )
                )
                
                # Strip markdown code blocks if the model wrapped the query
                sql = response.text.strip()
                sql = re.sub(r"^```(sql)?\s*", "", sql, flags=re.IGNORECASE)
                sql = re.sub(r"\s*```$", "", sql)
                return sql.strip()

            except ServerError as e:
                # 503 Overload or server hiccup: back off and retry next
                last_error = e
                time.sleep(1.5)
            except ClientError as e:
                # 404 or unsupported model name: skip directly to next model
                last_error = e
                break

    raise RuntimeError(f"All models were unavailable or encountered errors. Last error: {last_error}")

In [ ]:
# B2 — Prototype: Natural-Language Financial Database Queries

def ask_financial_db(user_question: str):
    print(f"User Query: {user_question}\n")
    
    # 1. Generate SQL
    generated_query = generate_sql(user_question)
    print(f"Generated SQL:\n{generated_query}\n")
    
    # 2. Execute safely via our read-only engine
    result_df = execute_safe_query(generated_query)
    return result_df

# Test 1: Aggregation & Join across companies and credit risk
question_1 = "Which company has the highest credit score, and what is its rating agency and risk category?"
ask_financial_db(question_1)
question_2 = "Which company had the highest revenue in Q3 2025, and what was its net income and operating margin percentage?"
ask_financial_db(question_2)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Unstructured financial audit notes and credit policy excerpts
DOCUMENTS = [
    {
        "doc_id": "DOC-101",
        "company": "Novastone Logistics",
        "title": "Q3 2025 Credit Risk & Debt Covenant Review",
        "content": "Novastone Logistics maintains a credit rating of 720 (Moody's). The company has complied with all debt covenants for Q3 2025. Working capital facility renewals are scheduled for Q1 2026, with interest rate margins pegged at SOFR + 180 bps."
    },
    {
        "doc_id": "DOC-102",
        "company": "Apex Cloud Systems",
        "title": "FY2025 Enterprise Cloud Expansion Policy",
        "content": "Apex Cloud Systems holds an S&P rating of 810 (Minimal Risk). All capital expenditures exceeding $5M require dual-signoff from the Risk Committee. Current liquidity ratios exceed covenant minimums by 2.4x."
    },
    {
        "doc_id": "DOC-103",
        "company": "Vantage Retail Group",
        "title": "Turnaround Strategy & Inventory Impairment Audit",
        "content": "Vantage Retail Group was rated 590 by Fitch (Moderate Risk) following Q2 operating losses. The advisory board mandated a 12% reduction in non-essential SG&A expenses and initiated renegotiation of long-term store lease agreements."
    }
]

def get_text_embedding(text: str) -> np.ndarray:
    """
    Generates text embeddings using gemini-embedding-001 (fallback to gemini-embedding-2).
    """
    models_to_try = ["gemini-embedding-001", "gemini-embedding-2"]
    last_err = None
    
    for m in models_to_try:
        try:
            response = client.models.embed_content(
                model=m,
                contents=text
            )
            # Handle SDK response structure
            if hasattr(response, "embeddings") and response.embeddings:
                return np.array(response.embeddings[0].values)
            elif hasattr(response, "embedding"):
                return np.array(response.embedding.values)
        except Exception as e:
            last_err = e
            continue
            
    raise RuntimeError(f"Embedding generation failed: {last_err}")

# Generate embeddings for the document repository
print("Generating vector embeddings for unstructured documents...")
for doc in DOCUMENTS:
    doc["embedding"] = get_text_embedding(doc["content"])
    print(f"Embedded: {doc['doc_id']} - {doc['company']}")

print(f"\nSuccessfully embedded {len(DOCUMENTS)} policy documents!")

In [ ]:
def retrieve_relevant_documents(user_query: str, top_k: int = 1):
    """
    Finds the most relevant policy document for a given user query using Cosine Similarity.
    """
    query_vector = get_text_embedding(user_query).reshape(1, -1)
    
    scored_docs = []
    for doc in DOCUMENTS:
        doc_vector = doc["embedding"].reshape(1, -1)
        sim_score = cosine_similarity(query_vector, doc_vector)[0][0]
        scored_docs.append((sim_score, doc))
        
    # Sort by highest similarity score
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    return scored_docs[:top_k]

# Test retrieval
test_query = "What covenants or liquidity requirements apply to cloud infrastructure investments?"
top_matches = retrieve_relevant_documents(test_query, top_k=1)

for score, doc in top_matches:
    print(f"Match Score: {score:.4f}")
    print(f"Company: {doc['company']}")
    print(f"Document Title: {doc['title']}")
    print(f"Excerpt: {doc['content']}")

In [ ]:
import time
from google.genai.errors import ServerError, ClientError

SYNTHESIS_MODELS = [
    "gemini-3.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "gemini-3.8-flash"
]

def generate_synthesis(prompt: str) -> str:
    """
    Synthesizes executive summary with automatic fallback across supported models.
    """
    last_err = None
    for model_name in SYNTHESIS_MODELS:
        for attempt in range(2):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(temperature=0.1)
                )
                return response.text.strip()
            except ServerError as e:
                last_err = e
                time.sleep(1.5)
            except ClientError as e:
                last_err = e
                break
    raise RuntimeError(f"All synthesis models failed. Last error: {last_err}")

def answer_hybrid_query(user_question: str) -> dict:
    """
    Hybrid decision engine: fetches structured data via SQL,
    retrieves policy context via embeddings, and synthesizes an evidence-grounded response.
    """
    print(f"--- Processing Query: {user_question} ---\n")
    
    # 1. Structured Retrieval (SQL)
    sql_query = generate_sql(user_question)
    structured_data = execute_safe_query(sql_query)
    
    # Format structured data for prompt context
    if isinstance(structured_data, pd.DataFrame):
        db_records_str = structured_data.to_dict(orient="records")
    else:
        db_records_str = str(structured_data)
        
    # 2. Unstructured Retrieval (Vector Search)
    retrieved_docs = retrieve_relevant_documents(user_question, top_k=1)
    best_doc = retrieved_docs[0][1]
    doc_context = f"Document [{best_doc['doc_id']}] {best_doc['title']}: {best_doc['content']}"
    
    # 3. Evidence-Grounded Synthesis Prompt
    synthesis_prompt = f"""
You are a Financial Decision Intelligence Assistant.
Synthesize a concise, grounded executive summary answering the user's question using ONLY the provided evidence.

Structured DB Evidence:
{db_records_str}

Unstructured Policy Evidence:
{doc_context}

User Question:
{user_question}

Instructions:
- Provide a direct answer.
- Explicitly cite the numbers from the database.
- Explicitly cite the policy rules or document ID from the unstructured text.
- Never extrapolate or assume facts not present in the evidence.
"""

    analysis_text = generate_synthesis(synthesis_prompt)
    
    return {
        "user_question": user_question,
        "structured_sql": sql_query,
        "structured_data": db_records_str,
        "retrieved_document": best_doc["doc_id"],
        "final_analysis": analysis_text
    }

In [ ]:
query = "What was Apex Cloud Systems' revenue and credit standing in Q3 2025, and what are their rules regarding capital expenditures?"
output = answer_hybrid_query(query)

print("Generated SQL:\n", output["structured_sql"])
print("\nStructured Data:\n", output["structured_data"])
print("\nRetrieved Document:\n", output["retrieved_document"])
print("\n--- Executive Summary ---")
print(output["final_analysis"])

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Any

class StructuredMetric(BaseModel):
    name: str
    revenue_usd: Optional[float] = None
    net_income_usd: Optional[float] = None
    credit_score: Optional[int] = None
    risk_category: Optional[str] = None
    rating_agency: Optional[str] = None
    default_probability: Optional[float] = None

class EvidenceSource(BaseModel):
    source_type: str = Field(description="'relational_db' or 'unstructured_policy'")
    source_id: str
    details: Any

class DecisionIntelligenceResponse(BaseModel):
    query: str
    status: str = "success"
    structured_metrics: List[StructuredMetric]
    retrieved_policy_id: str
    evidence_trail: List[EvidenceSource]
    executive_summary: str

def format_as_pydantic_response(raw_output: dict) -> DecisionIntelligenceResponse:
    """
    Validates and formats raw hybrid pipeline output into a strict Pydantic model.
    """
    # 1. Parse structured metrics
    records = raw_output.get("structured_data", [])
    if isinstance(records, list):
        metrics = [StructuredMetric(**item) for item in records]
    else:
        metrics = []

    # 2. Build the evidence audit trail
    evidence = [
        EvidenceSource(
            source_type="relational_db",
            source_id="sqlite_financial_data",
            details={"sql": raw_output.get("structured_sql")}
        ),
        EvidenceSource(
            source_type="unstructured_policy",
            source_id=raw_output.get("retrieved_document", "UNKNOWN"),
            details={"match": "cosine_similarity_top_1"}
        )
    ]

    # 3. Assemble and validate
    validated_payload = DecisionIntelligenceResponse(
        query=raw_output["user_question"],
        structured_metrics=metrics,
        retrieved_policy_id=raw_output["retrieved_document"],
        evidence_trail=evidence,
        executive_summary=raw_output["final_analysis"]
    )
    return validated_payload

In [ ]:
# Pass the output from your previous cell
validated_data = format_as_pydantic_response(output)

# Display the validated JSON payload
print(validated_data.model_dump_json(indent=2))

# C. Main Application

This section contains the primary implementation of the Financial Decision Intelligence Assistant.

It includes the application source code, FastAPI server startup, and server shutdown.

- **C1 — Main Application:** Writes the application code to `main.py`.
- **C2 — Start Server:** Starts the FastAPI server for API requests.
- **C3 — Stop Server:** Shuts down the running server gracefully.

In [ ]:
%%writefile main.py
# C1 — Main Application: Financial Decision Intelligence API


import os
import re
import time
import sqlite3
import numpy as np
import pandas as pd
from typing import List, Optional, Any
from pandas.errors import DatabaseError
from sklearn.metrics.pairwise import cosine_similarity
from pydantic import BaseModel, Field
from fastapi import FastAPI, HTTPException
from google import genai
from google.genai import types

# 1. Initialize API Client
client = genai.Client()

app = FastAPI(
    title="Financial Decision Intelligence Assistant API",
    description="Hybrid Text-to-SQL and RAG assistant with read-only SQLite guardrails.",
    version="1.0.0"
)

# 2. Database Schema Context
DB_PATH = "financial_data.db"
DB_SCHEMA_CONTEXT = """
Database Schema:
1. companies (company_id INTEGER PRIMARY KEY, ticker TEXT UNIQUE, name TEXT, sector TEXT);
2. quarterly_financials (financial_id INTEGER PRIMARY KEY, company_id INTEGER, quarter TEXT, fiscal_year INTEGER, revenue_usd REAL, net_income_usd REAL, operating_expense_usd REAL);
3. credit_risk (risk_id INTEGER PRIMARY KEY, company_id INTEGER, rating_agency TEXT, credit_score INTEGER, risk_category TEXT, default_probability REAL);
"""

SYSTEM_PROMPT = f"""
You are a specialized SQL assistant for a financial analytics database.
Given a user's question, output ONLY a valid SQLite read-only query using the provided schema.

Rules:
1. Return ONLY the raw SQL query. No markdown formatting (no ```sql or ```).
2. Only write SELECT statements. Never write INSERT, UPDATE, DELETE, DROP, or ALTER.
3. Correctly perform JOIN operations on `company_id`.

{DB_SCHEMA_CONTEXT}
"""

# 3. Policy Document Repository
DOCUMENTS = [
    {
        "doc_id": "DOC-101",
        "company": "Novastone Logistics",
        "title": "Q3 2025 Credit Risk & Debt Covenant Review",
        "content": "Novastone Logistics maintains a credit rating of 720 (Moody's). The company has complied with all debt covenants for Q3 2025. Working capital facility renewals are scheduled for Q1 2026, with interest rate margins pegged at SOFR + 180 bps."
    },
    {
        "doc_id": "DOC-102",
        "company": "Apex Cloud Systems",
        "title": "FY2025 Enterprise Cloud Expansion Policy",
        "content": "Apex Cloud Systems holds an S&P rating of 810 (Minimal Risk). All capital expenditures exceeding $5M require dual-signoff from the Risk Committee. Current liquidity ratios exceed covenant minimums by 2.4x."
    },
    {
        "doc_id": "DOC-103",
        "company": "Vantage Retail Group",
        "title": "Turnaround Strategy & Inventory Impairment Audit",
        "content": "Vantage Retail Group was rated 590 by Fitch (Moderate Risk) following Q2 operating losses. The advisory board mandated a 12% reduction in non-essential SG&A expenses and initiated renegotiation of long-term store lease agreements."
    }
]

# 4. Core Processing Helpers
def get_text_embedding(text: str) -> np.ndarray:
    models_to_try = ["gemini-embedding-001", "gemini-embedding-2"]
    for m in models_to_try:
        try:
            res = client.models.embed_content(model=m, contents=text)
            if hasattr(res, "embeddings") and res.embeddings:
                return np.array(res.embeddings[0].values)
            elif hasattr(res, "embedding"):
                return np.array(res.embedding.values)
        except Exception:
            continue
    raise RuntimeError("Failed to generate embedding.")

# Precompute document vectors on startup
for doc in DOCUMENTS:
    doc["embedding"] = get_text_embedding(doc["content"])

def execute_safe_query(query: str, db_path: str = DB_PATH):
    uri = f"file:{db_path}?mode=ro"
    try:
        conn = sqlite3.connect(uri, uri=True)
        conn.execute("PRAGMA query_only = ON;")
        df = pd.read_sql_query(query, conn)
        conn.close()
        return df
    except (sqlite3.OperationalError, DatabaseError, Exception) as e:
        raise HTTPException(status_code=400, detail=f"Database Guardrail Violation: {e}")

# USING THE EXACT MODELS THAT WORKED IN YOUR NOTEBOOK
WORKING_MODELS = ["gemini-3.5-flash-lite","gemini-3.8-flash", "gemini-3.1-pro-preview"]

def generate_sql(question: str) -> str:
    last_err = None
    for m in WORKING_MODELS:
        try:
            res = client.models.generate_content(
                model=m,
                contents=question,
                config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0.0)
            )
            sql = res.text.strip()
            sql = re.sub(r"^```(sql)?\s*", "", sql, flags=re.IGNORECASE)
            return re.sub(r"\s*```$", "", sql).strip()
        except Exception as e:
            last_err = e
            error_text = str(e)

            if "429" in error_text or "404" in error_text:
                break

            time.sleep(1)
            continue
    raise HTTPException(status_code=503, detail=f"Text-to-SQL error: {last_err}")

def retrieve_top_document(query: str, threshold: float = 0.60):
    q_vec = get_text_embedding(query).reshape(1, -1)
    scored = []
    for doc in DOCUMENTS:
        d_vec = doc["embedding"].reshape(1, -1)
        similarity = cosine_similarity(q_vec, d_vec)[0][0]
        scored.append((similarity, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    best_score, best_doc = scored[0]
    if best_score < threshold:
        return None
    return best_doc

def synthesize_grounded_answer(user_question: str, db_data: list, doc: Optional[dict]) -> str:
    # Safe check: handle queries where no policy met the similarity threshold
    if doc is not None:
        doc_context = f"Document [{doc['doc_id']}] {doc['title']}: {doc['content']}"
    else:
        doc_context = "No relevant corporate policy documents found matching the similarity threshold."

    prompt = f"""
You are a Financial Decision Intelligence Assistant.
Synthesize a concise, grounded executive summary answering the question using ONLY the provided evidence.

Structured DB Evidence:
{db_data}

Unstructured Policy Evidence:
{doc_context}

Question:
{user_question}

Instructions:
- Provide a direct answer.
- Explicitly cite database metrics if present.
- Explicitly cite the policy rules or Document ID if policy evidence is provided.
- If no relevant policy was found, state that clearly without guessing.
- Do not extrapolate beyond the provided text.
"""
    last_err = None
    for m in WORKING_MODELS:
        try:
            res = client.models.generate_content(
                model=m,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.1)
            )
            return res.text.strip()
        except Exception as e:
            last_err = e
            time.sleep(1)
            continue
    raise HTTPException(status_code=503, detail=f"Synthesis error: {last_err}")

# 5. Pydantic Schemas
class QueryRequest(BaseModel):
    question: str

class StructuredMetric(BaseModel):
    name: Optional[str] = None
    ticker: Optional[str] = None
    quarter: Optional[str] = None
    fiscal_year: Optional[int] = None
    revenue_usd: Optional[float] = None
    net_income_usd: Optional[float] = None
    operating_expense_usd: Optional[float] = None
    credit_score: Optional[int] = None
    risk_category: Optional[str] = None
    rating_agency: Optional[str] = None
    default_probability: Optional[float] = None

class EvidenceSource(BaseModel):
    source_type: str
    source_id: str
    details: Any

class QueryResponse(BaseModel):
    query: str
    status: str = "success"
    structured_metrics: List[StructuredMetric]
    retrieved_policy_id: str
    evidence_trail: List[EvidenceSource]
    executive_summary: str

# 6. API Endpoints
@app.get("/")
def health_check():
    return {"status": "healthy", "service": "Financial Decision Intelligence Assistant API"}

@app.post("/query", response_model=QueryResponse)
def handle_query(request: QueryRequest):
    # Step A: SQL Generation & Execution
    sql_query = generate_sql(request.question)
    df_result = execute_safe_query(sql_query)
    records = df_result.to_dict(orient="records")

    # Step B: Policy Vector Retrieval
    best_doc = retrieve_top_document(request.question)

    # Step C: Synthesize Grounded Output
    executive_summary = synthesize_grounded_answer(request.question, records, best_doc)

    # Step D: Package into Pydantic Contract
    metrics = [StructuredMetric(**r) for r in records]
    evidence = [
        EvidenceSource(
            source_type="relational_db",
            source_id="sqlite_financial_data",
            details={"sql": sql_query},
        )
    ]

    if best_doc is not None:
        evidence.append(
            EvidenceSource(
                source_type="unstructured_policy",
                source_id=best_doc["doc_id"],
                details={"title": best_doc["title"]},
            )
        )
    return QueryResponse(
        query=request.question,
        structured_metrics=metrics,
        retrieved_policy_id=best_doc["doc_id"] if best_doc else "NOT_RETRIEVED",
        evidence_trail=evidence,
        executive_summary=executive_summary,
    )

In [ ]:
# C2 — Start the FastAPI Server

import asyncio
import uvicorn
from main import app

# Start Uvicorn directly on Jupyter's existing event loop
config = uvicorn.Config(app=app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

# Run server as a non-blocking background task
server_task = asyncio.create_task(server.serve())

print("=" * 60)
print("Server is LIVE!")
print("Open this link in your browser:")
print("👉  http://127.0.0.1:8000/docs")
print("=" * 60)

In [ ]:
server.should_exit = True
await server_task
print("Server stopped successfully.")

In [ ]:
# D. Testing and Validation

#This section contains manual verification tests for the Financial Decision Intelligence Assistant.

#Run these tests when validating code changes or investigating problems. They are not required for every normal application run.

In [ ]:
#Test 1 — Structured Output Schema
import importlib
import main

importlib.reload(main)

print("Ticker supported:",
      "ticker" in main.StructuredMetric.model_fields)

print("Quarter supported:",
      "quarter" in main.StructuredMetric.model_fields)

print("Operating expense supported:",
      "operating_expense_usd" in main.StructuredMetric.model_fields)

In [ ]:
#Test 2 — Document Retrieval
import importlib
import main

importlib.reload(main)

test_questions = [
    "What are Apex Cloud Systems' capital expenditure approval rules?",
    "What is the employee vacation policy?",
]

for question in test_questions:
    result = main.retrieve_top_document(question)

    print("\nQuestion:", question)

    if result is None:
        print("No sufficiently relevant document found.")
    else:
        print("Retrieved:", result["doc_id"])
        print("Title:", result["title"])

In [ ]:
#Test 3 — End-to-End Integration
import importlib
import main

importlib.reload(main)

question = (
    "What was Apex Cloud Systems' revenue and credit standing "
    "in Q3 2025, and what are their rules regarding capital expenditures?"
)

try:
    result = main.handle_query(main.QueryRequest(question=question))

    print("Status:", result.status)
    print("Policy ID:", result.retrieved_policy_id)
    print("\nExecutive Summary:")
    print(result.executive_summary)
    print("\nStructured Metrics:")
    for metric in result.structured_metrics:
        print(metric.model_dump(exclude_none=True))
    print("\nEvidence Trail:")
    for evidence in result.evidence_trail:
        print(evidence.model_dump())

except Exception as e:
    print("FAILED:", repr(e))

In [ ]:
# Test 4 — Database Error Handling
from fastapi import HTTPException

try:
    main.execute_safe_query("SELECT * FROM nonexistent_table")
except HTTPException as e:
    print("HTTP status:", e.status_code)
    print("Error message:", e.detail)

In [ ]:
#Test 5 — Read-Only Database Protection
try:
    main.execute_safe_query("DELETE FROM companies")
    print("WARNING: Query was not rejected!")
except Exception as e:
    print("Query rejected:", e)

In [ ]:
# Diagnostic — Inspect SQL Generation
import inspect
print(inspect.getsource(main.generate_sql))

In [ ]:
#Diagnostic — Document Similarity Scores
import main
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

test_questions = [
    "What are Apex Cloud Systems' capital expenditure approval rules?",
    "What is the employee vacation policy?",
]

for question in test_questions:
    query_vector = main.get_text_embedding(question).reshape(1, -1)

    print(f"\nQuestion: {question}")

    for doc in main.DOCUMENTS:
        document_vector = doc["embedding"].reshape(1, -1)

        score = cosine_similarity(
            query_vector,
            document_vector
        )[0][0]

        print(f"{doc['doc_id']}: {score:.4f}")